## Run

In [4]:
# TODO: Develop your agents under `agentic/agents`
# TODO: Develop your tools under `agentic/tools`
# TODO: Modify `agentic/workflow` in order to orchestrate your agents

In [2]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator

from agentic.workflow import orchestrator
from utils import chat_interface

In [3]:
chat_interface(orchestrator, "1")

User: cielak
Assistant: I want to make sure you get the best possible help. I'm connecting you with a specialist who has more expertise in this area. They'll follow up with you soon.
User: how are you today
Assistant: I want to make sure you get the best possible help. I'm connecting you with a specialist who has more expertise in this area. They'll follow up with you soon.
User: 
Assistant: I want to make sure you get the best possible help. I'm connecting you with a specialist who has more expertise in this area. They'll follow up with you soon.
User: 


KeyboardInterrupt: 

## Testing & Inspection

**Test the orchestrator with a sample message**

⏱️ **Performance Note**: 
- First query: ~10-15s (initializes embeddings and builds cache)
- Subsequent queries: ~2-5s (uses cached embeddings)

In [ ]:
from langchain_core.messages import HumanMessage

# Test with a simple query
test_state = {
    "messages": [HumanMessage(content="I can't log in to my account")],
}

config = {
    "configurable": {
        "thread_id": "test-1",
    }
}

result = orchestrator.invoke(test_state, config)

# Find the last non-system message (skip supervisor/internal messages)
final_response = None
for msg in reversed(result["messages"]):
    if hasattr(msg, 'name') and msg.name in ['resolver', 'escalation']:
        final_response = msg.content
        break
    elif not hasattr(msg, 'name') or (hasattr(msg, 'name') and msg.name not in ['supervisor', 'classifier', 'tool_agent']):
        final_response = msg.content
        break

print("Final response:", final_response if final_response else result["messages"][-1].content)

✅ Loaded cached knowledge base from /home/c/Nauka/final-project/starter/data/core/faiss_cache_cultpass


KeyboardInterrupt: 

**Inspect the workflow graph**

In [1]:
# View the graph structure
try:
    from IPython.display import Image, display
    display(Image(orchestrator.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Could not display graph: {e}")
    print("Graph nodes:", orchestrator.get_graph().nodes)

Could not display graph: name 'orchestrator' is not defined


NameError: name 'orchestrator' is not defined

**Check available agents and tools**

In [3]:
from agentic.agents.tool_agent import ToolAgent

tool_agent = ToolAgent()
print("Available tools:")
for tool in tool_agent.get_available_tools():
    print(f"  - {tool}")

Available tools:
  - get_user_info
  - get_subscription_status
  - get_reservations
  - cancel_reservation
  - check_account_status
  - request_refund


**View conversation history for a thread**

In [4]:
# Retrieve state history for a specific thread
thread_id = "test-1"
config = {"configurable": {"thread_id": thread_id}}

try:
    # get_state_history returns an iterator
    state_history = orchestrator.get_state_history(config)
    print(f"State history for thread '{thread_id}':")
    
    # Iterate through the history (limit to 5 most recent states)
    for i, state in enumerate(state_history):
        if i >= 5:  # Show only last 5 states
            break
        print(f"\nState {i}:")
        print(f"  Checkpoint ID: {state.config.get('configurable', {}).get('checkpoint_id', 'N/A')}")
        if state.values.get("messages"):
            print(f"  Messages: {len(state.values['messages'])}")
            if state.values['messages']:
                last_msg = state.values['messages'][-1]
                content_preview = last_msg.content[:100] if hasattr(last_msg, 'content') else str(last_msg)[:100]
                print(f"  Last message: {content_preview}...")
        if state.values.get("next_agent"):
            print(f"  Next agent: {state.values['next_agent']}")
except Exception as e:
    print(f"Could not retrieve state history: {e}")
    import traceback
    traceback.print_exc()

State history for thread 'test-1':
Could not retrieve state history: ext_hook failed


Traceback (most recent call last):
  File "/tmp/ipykernel_192271/1830512228.py", line 11, in <module>
    for i, state in enumerate(state_history):
                    ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/c/Nauka/final-project/venv/lib/python3.12/site-packages/langgraph/pregel/main.py", line 1357, in get_state_history
    for checkpoint_tuple in list(
                            ^^^^^
  File "/home/c/Nauka/final-project/venv/lib/python3.12/site-packages/langgraph/checkpoint/sqlite/__init__.py", line 357, in list
    self.serde.loads_typed((type, checkpoint)),
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/c/Nauka/final-project/venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/jsonplus.py", line 202, in loads_typed
    return ormsgpack.unpackb(
           ^^^^^^^^^^^^^^^^^^
ValueError: ext_hook failed


**Get current state of a thread**

In [ ]:
# Get the current state of a specific thread
thread_id = "test-1"
config = {"configurable": {"thread_id": thread_id}}

try:
    current_state = orchestrator.get_state(config)
    print(f"Current state for thread '{thread_id}':")
    print(f"  Values keys: {list(current_state.values.keys())}")
    
    if current_state.values.get("messages"):
        print(f"  Total messages: {len(current_state.values['messages'])}")
        print(f"  Last message: {current_state.values['messages'][-1].content[:150]}...")
    
    if current_state.values.get("classification"):
        print(f"  Classification: {current_state.values['classification'].get('classification', {}).get('category', 'N/A')}")
    
    if current_state.values.get("next_agent"):
        print(f"  Next agent: {current_state.values['next_agent']}")
    
    if current_state.values.get("escalation_needed"):
        print(f"  Escalation needed: {current_state.values['escalation_needed']}")
        
    print(f"\n  Next step: {current_state.next}")
except Exception as e:
    print(f"Could not retrieve current state: {e}")